# Visits - Silver Layer (fact_visits)

Ingestion incrémentale depuis Bronze avec enrichissement par jointures sur les tables dimensions Silver (patients, hospitals, diagnosis). Utilise `foreachBatch` + `MERGE` pour dédupliquer sur `visit_id`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG    = "hospital"
SOURCE     = f"{CATALOG}.bronze.visit_raw"
TARGET     = f"{CATALOG}.silver.fact_visit"
QUARANTINE = f"{CATALOG}.silver.quarantine_visit"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/checkpoints/silver/fact_visit"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET} (
  visit_id        STRING NOT NULL,
  patient_id      STRING,
  hospital_id     STRING,
  diagnosis_code  STRING,
  admission_date  DATE,
  discharge_date  DATE,
  length_of_stay  INT,
  cost            DECIMAL(12,2),
  _source_file    STRING,
  _updated_at     TIMESTAMP
) COMMENT 'Fait visite : clés et mesures uniquement'
""")

In [0]:
def upsert_fact_visit(batch_df, batch_id):
    # 1. Typage (try_ : NULL si la valeur est invalide, au lieu de planter)
    typed = batch_df.select(
        F.trim("visit_id").alias("visit_id"),
        F.trim("patient_id").alias("patient_id"),
        F.trim("hospital_id").alias("hospital_id"),
        F.upper(F.trim("diagnosis_code")).alias("diagnosis_code"),
        F.expr("try_to_date(admission_date, 'yyyy-MM-dd')").alias("admission_date"),
        F.expr("try_to_date(discharge_date, 'yyyy-MM-dd')").alias("discharge_date"),
        F.expr("try_cast(cost AS DECIMAL(12,2))").alias("cost"),
        "_rescued_data", "_source_file", "_ingested_at")

    # 2. Règles qualité, avec la raison du rejet
    reason = (F.when(F.col("visit_id").isNull(), "visit_id manquant")
               .when(F.col("admission_date").isNull() | F.col("discharge_date").isNull(), "date invalide")
               .when(F.col("discharge_date") < F.col("admission_date"), "sortie avant admission")
               .when(F.col("cost").isNull() | (F.col("cost") < 0), "coût invalide")
               .when(F.col("_rescued_data").isNotNull(), "données inattendues"))
    checked = typed.withColumn("_reject_reason", reason)

    # 3. Quarantaine (idempotente)
    (checked.filter("_reject_reason IS NOT NULL")
        .withColumn("_rejected_at", F.current_timestamp())
        .write.mode("append")
        .option("txnAppId", "quarantine_visit").option("txnVersion", batch_id)
        .saveAsTable(QUARANTINE))

    # 4. Dédoublonnage : la version la plus récente par visite
    w = Window.partitionBy("visit_id").orderBy(F.col("_ingested_at").desc())
    latest = (checked.filter("_reject_reason IS NULL")
        .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
        .withColumn("length_of_stay", F.datediff("discharge_date", "admission_date"))
        .withColumn("_updated_at", F.current_timestamp())
        .select("visit_id", "patient_id", "hospital_id", "diagnosis_code",
                "admission_date", "discharge_date", "length_of_stay", "cost",
                "_source_file", "_updated_at"))

    # 5. MERGE incrémental
    (DeltaTable.forName(spark, TARGET).alias("t")
        .merge(latest.alias("s"), "t.visit_id = s.visit_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
(spark.readStream.table(SOURCE)
    .writeStream.foreachBatch(upsert_fact_visit)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start().awaitTermination())

display(spark.table(TARGET).orderBy("visit_id"))